# Preliminary Colab adaptation: COLING 2025 joint intent + slot filling

**Original paper:** Maia Aguirre et al., *Fine-Tuning Medium-Scale LLMs for Joint Intent Classification and Slot Filling: A Data-Efficient and Cost-Effective Solution for SMEs*, COLING Industry Track 2025, pp. 251–262.

## Honest status
The earlier local TF-IDF experiment is retained only as a **preliminary concept demonstration** in `results.json`; it is not an implementation of this paper. This notebook now contains a Colab-oriented path that captures the central method more faithfully: a small public encoder, a shared encoder with intent and BIO slot heads, and LoRA adapters. It is **not executed in this workspace** because `torch`, `transformers`, `datasets`, and `peft` are unavailable in the local runtime. Do not treat this notebook as completed evidence until its Colab cells run successfully and save outputs.

In [ ]:
!pip -q install "transformers>=4.45" "datasets>=2.20" "peft>=0.12" "seqeval>=1.2" accelerate

In [ ]:
import re, json, random, numpy as np, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, DataCollatorForTokenClassification
from peft import LoraConfig, TaskType, get_peft_model

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
MODEL_ID="google/bert_uncased_L-2_H-128_A-2"
# MASSIVE provides utterances, intents, and annotated slot markup. Keep three intents and a small split for free Colab.
raw=load_dataset("AmazonScience/massive", "en-US")
keep_intents=sorted(set(raw["train"]["intent"]))[:3]
train=raw["train"].filter(lambda r:r["intent"] in keep_intents).shuffle(seed=SEED).select(range(min(240, len(raw["train"]))))
valid=raw["validation"].filter(lambda r:r["intent"] in keep_intents).shuffle(seed=SEED).select(range(min(90, len(raw["validation"]))))
print(raw, keep_intents, len(train), len(valid))

In [ ]:
def parse_annotated(text):
    parts=re.split(r"(<[^>]+>)", text); tokens=[]; labels=[]; slot=None
    for part in parts:
        if not part: continue
        if part.startswith("<") and part.endswith(">"):
            if part.startswith("</"): slot=None
            else: slot=part[1:-1]
            continue
        for tok in part.split():
            tokens.append(tok); labels.append("B-"+slot if slot else "O"); slot = slot if slot else None
            if slot: labels[-1]="I-"+slot
    # Repair the first token of each contiguous slot span to B-.
    for i,label in enumerate(labels):
        if label.startswith("I-") and (i==0 or labels[i-1] != label): labels[i]="B-"+label[2:]
    return tokens,labels

examples=[parse_annotated(x) for x in train["annot_utt"]]
intent_names=sorted(set(train["intent"])); intent2id={x:i for i,x in enumerate(intent_names)}
slot_names=sorted({lab for _,labs in examples for lab in labs}); slot2id={x:i for i,x in enumerate(["O"]+[x for x in slot_names if x!="O"])}
print("intents",intent2id,"slots",slot2id)

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

def encode(row):
    words,slots=parse_annotated(row["annot_utt"])
    enc=tokenizer(words,is_split_into_words=True,truncation=True,max_length=64)
    word_ids=enc.word_ids(); aligned=[]
    for wid in word_ids: aligned.append(-100 if wid is None else slot2id.get(slots[wid],slot2id["O"]))
    enc["slot_labels"]=aligned; enc["intent_labels"]=intent2id[row["intent"]]; return enc

train_enc=train.map(encode,remove_columns=train.column_names); valid_enc=valid.map(encode,remove_columns=valid.column_names)
print(train_enc[0].keys())

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader

class JointIntentSlot(nn.Module):
    def __init__(self):
        super().__init__(); self.encoder=AutoModel.from_pretrained(MODEL_ID)
        hidden=self.encoder.config.hidden_size
        self.intent_head=nn.Linear(hidden,len(intent2id)); self.slot_head=nn.Linear(hidden,len(slot2id))
    def forward(self,input_ids,attention_mask,slot_labels=None,intent_labels=None):
        h=self.encoder(input_ids=input_ids,attention_mask=attention_mask).last_hidden_state
        pooled=(h*attention_mask.unsqueeze(-1)).sum(1)/attention_mask.sum(1,keepdim=True)
        il=self.intent_head(pooled); sl=self.slot_head(h); loss=None
        if intent_labels is not None:
            loss=nn.CrossEntropyLoss()(il,intent_labels)+nn.CrossEntropyLoss(ignore_index=-100)(sl.view(-1,sl.size(-1)),slot_labels.view(-1))
        return loss,il,sl

model=JointIntentSlot()
config=LoraConfig(task_type=TaskType.FEATURE_EXTRACTION,r=4,lora_alpha=8,lora_dropout=0.1,target_modules=["query","value"])
model.encoder=get_peft_model(model.encoder,config)
print("trainable parameters",sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Method path defined; run the next training/evaluation cell in Colab after reviewing the small split.")

## Training/evaluation status

The model/data preparation above is the faithful part of this adaptation: it uses a small public encoder, LoRA adapters, joint intent and BIO slot heads, and MASSIVE annotations. A full training loop is intentionally not claimed as executed here. If Colab resources permit, add a short AdamW loop, evaluate intent accuracy and slot entity-F1 on `valid_enc`, and save `results.json` with the exact seed, model ID, split sizes, and metrics. Until then this remains **preliminary / not implemented** and must not be listed as a completed paper implementation.